# DeepSeek-R1 QLoRA fine-tuning with TRL

The dataset is converted to TRL's prompt-completion format. TRL computes loss on the completion only.

In [ ]:
%%capture
!pip install -U unsloth trl

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
MAX_SEQ_LENGTH = 2112

print(f"Loading {MODEL_NAME} in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=True,
)
print("Model loaded.")

In [ ]:
print("Adding LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA adapters ready.")

In [ ]:
from datasets import load_dataset, concatenate_datasets

DATA_PATH = "/kaggle/input/datasets/atomstack001/bioreasoning-sft-trial-data/sft_rejection_sampled_train_phase_one.jsonl"

print(f"Loading SFT data from {DATA_PATH}...")
raw_dataset = load_dataset("json", data_files=str(DATA_PATH), split="train")
print(f"Loaded {len(raw_dataset)} rows.")


def select_by_label(source, counts, seed):
    selected = []
    for label, count in counts.items():
        rows = source.filter(
            lambda example: example["label"] == label,
            desc=f"Selecting {label} rows",
        )
        selected.append(rows.shuffle(seed=seed).select(range(count)))
        print(f"Selected {count} '{label}' rows.")
    return concatenate_datasets(selected).shuffle(seed=seed)


train_source = raw_dataset.filter(
    lambda example: example["split"] == "train",
    desc="Reading train split",
)
validation_source = raw_dataset.filter(
    lambda example: example["split"] == "val",
    desc="Reading validation split",
)

train_dataset = select_by_label(
    train_source,
    {"none": 1112, "up": 603, "down": 285},
    seed=42,
)
validation_dataset = select_by_label(
    validation_source,
    {"none": 122, "up": 66, "down": 32},
    seed=43,
)


def to_prompt_completion(example):
    """Preserve DeepSeek reasoning while using TRL prompt-completion SFT."""
    messages = example["messages"]
    assistant = messages[-1]
    if assistant["role"] != "assistant":
        raise ValueError("The final message must be an assistant response.")

    completion_text = assistant["content"].strip()
    if not completion_text.startswith("<think>"):
        completion_text = f"<think>\n{completion_text}"

    system_text = "\n\n".join(
        message["content"].strip()
        for message in messages[:-1]
        if message["role"] == "system"
    )
    user_text = "\n\n".join(
        message["content"].strip()
        for message in messages[:-1]
        if message["role"] == "user"
    )

    prompt = (
        f"{system_text}"
        f"<｜User｜>{user_text}"
        f"<｜Assistant｜>"
    )
    completion = (
        f"{completion_text}"
        f"{tokenizer.eos_token or ''}"
    )
    return {"prompt": prompt, "completion": completion}


print("Formatting data as TRL prompt-completion pairs...")
train_dataset = train_dataset.map(
    to_prompt_completion,
    remove_columns=train_dataset.column_names,
    desc="Formatting train data",
)
validation_dataset = validation_dataset.map(
    to_prompt_completion,
    remove_columns=validation_dataset.column_names,
    desc="Formatting validation data",
)

print(f"Train rows: {len(train_dataset)}")
print(f"Validation rows: {len(validation_dataset)}")
print("Dataset fields:", train_dataset.column_names)
print("Sample prompt ending:", repr(train_dataset[0]["prompt"][-120:]))
print("Sample completion ending:", repr(train_dataset[0]["completion"][-120:]))

In [ ]:
from trl import SFTConfig, SFTTrainer

print("Creating TRL SFT trainer...")
training_args = SFTConfig(
    output_dir="/kaggle/working/llama_8b_adapter",
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    packing=False,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=1e-4,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=100,
    logging_first_step=True,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    seed=42,
    data_seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
)
print("Trainer ready. TRL will compute loss on completion tokens only.")

In [ ]:
print("Starting training...")
trainer_stats = trainer.train()

print("Evaluating best checkpoint...")
validation_metrics = trainer.evaluate()
print(f"Train loss: {trainer_stats.metrics.get('train_loss', float('nan')):.6f}")
print(f"Validation loss: {validation_metrics.get('eval_loss', float('nan')):.6f}")

save_path = "/kaggle/working/llama_8b_adapter/final_adapter"
print(f"Saving adapter to {save_path}...")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Training complete; adapter saved.")